<a href="https://colab.research.google.com/github/Ryan56-56/onixhw/blob/main/onix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split


In [22]:
csv_path = "/content/drive/MyDrive/data.csv"
df = pd.read_csv(csv_path, sep=';')

# Convert 'Target' column to numeric labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Target'] = le.fit_transform(df['Target'])

# Update input_cols and output_cols to match the actual dataframe columns
# Assuming 'Target' is the output column and the rest are input features
input_cols = [col for col in df.columns if col != 'Target']
output_cols = ['Target']

X = df[input_cols].values.astype(np.float32)
y = df[output_cols].values.astype(np.float32)

X.shape, y.shape

((4424, 36), (4424, 1))

In [20]:
print(df.columns.tolist())

['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance\t', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP', 'Target']


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [24]:
x_means = torch.tensor(X_train.mean(axis=0), dtype=torch.float32)
x_stds  = torch.tensor(X_train.std(axis=0) + 1e-8, dtype=torch.float32)

y_means = torch.tensor(y_train.mean(axis=0), dtype=torch.float32)
y_stds  = torch.tensor(y_train.std(axis=0) + 1e-8, dtype=torch.float32)

def scale_y(y):
    return (y - y_means) / y_stds


In [25]:
train_ds = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(scale_y(torch.tensor(y_train)))
)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)


/tmp/ipykernel_7711/1692203786.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(scale_y(torch.tensor(y_train)))


In [34]:
class ResidualNet_Forward(nn.Module):
    def __init__(self, x_means, x_stds, y_means, y_stds):
        super().__init__()
        self.x_means = x_means
        self.x_stds  = x_stds
        self.y_means = y_means
        self.y_stds  = y_stds

        # Correct the input dimension from 7 to 36 to match the data's 36 features
        self.fc1 = nn.Linear(36, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1) # Changed output dimension from 4 to 1
        self.act = nn.ReLU()

        # Correct the input dimension from 7 to 36 for the projection layer
        self.input_proj = nn.Linear(36, 64)

    def forward(self, x):
        x_scaled = (x - self.x_means) / self.x_stds
        x0 = self.input_proj(x_scaled)
        x1 = self.act(self.fc1(x_scaled))
        x2 = self.act(self.fc2(x1))
        x_res = x2 + x0
        y_scaled = self.fc3(x_res)
        y = y_scaled * self.y_stds + self.y_means
        return y

In [35]:
model = ResidualNet_Forward(x_means, x_stds, y_means, y_stds).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [36]:
model.train()

for epoch in range(200):
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)

        # The target 'yb' already contains scaled 'y_train', so we don't need to scale 'pred' again for loss calculation
        # Also, the model output 'pred' is already unscaled, so we need to scale it for comparison with 'yb'
        loss = loss_fn(scale_y(pred), yb)
        loss.backward()
        optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

Epoch 0 Loss 0.2668
Epoch 50 Loss 0.5527
Epoch 100 Loss 0.0879
Epoch 150 Loss 0.1264


In [31]:
model = ResidualNet_Forward(x_means, x_stds, y_means, y_stds).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [32]:
model.train()

for epoch in range(200):
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        pred = model(xb)

        loss = loss_fn((pred - y_means)/y_stds, yb)
        loss.backward()
        optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([32, 1])) that is different to the input size (torch.Size([32, 4])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([19, 1])) that is different to the input size (torch.Size([19, 4])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0 Loss 0.3886
Epoch 50 Loss 0.0513
Epoch 100 Loss 0.0858
Epoch 150 Loss 0.0866


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResidualNet_Forward(x_means, x_stds, y_means, y_stds).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()


In [42]:
model.eval()

dummy_input = torch.randn(1, 36).to(device)

onnx_path = "DLnet2026_WineData.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
)

print("ONNX model saved to:", onnx_path)

/tmp/ipykernel_7711/2160626399.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0418 02:00:46.615000 7711 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ResidualNet_Forward([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResidualNet_Forward([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX model saved to: DLnet2026_WineData.onnx


In [41]:
!pip install onnxscript

In [40]:
!pip install onnxscript